# TERF — ray tracing LOS/NLOS (stazione permanente)

Pipeline Colab per la stazione **TERF** (`TERF00CYP`, Cyprus).

**Repo corretto:** `https://github.com/andrea444460/gnss_gpu_los_nlos` branch `feature/cuda-preprocess-area-map`  
(non usare `rsasaki0109/gnss_gpu` — lì non c'è lo script TERF).

**Output** in `/content/terf_work/results/`:
- mesh OSM, labels CSV, **viewer 3D** `terf_los_viz_YYYYDOY.html` (raggi per satellite nel RINEX, come Odaiba)

**Runtime:** GPU (T4) + build CUDA `_bvh`.

In [ ]:
# 1) Verifica GPU
!nvidia-smi

In [ ]:
# 2) Clone / aggiorna repo ANDREA (obbligatorio)
import os, shutil, subprocess

REPO = "/content/gnss_gpu"
BRANCH = "feature/cuda-preprocess-area-map"
REPO_URL = "https://github.com/andrea444460/gnss_gpu_los_nlos.git"
SCRIPT = f"{REPO}/experiments/run_terf_permanent_station_colab.py"

def _has_script():
    return os.path.isfile(SCRIPT)

if os.path.isdir(REPO) and not _has_script():
    print("Cartella esiste ma manca lo script TERF → rimuovo clone errato")
    shutil.rmtree(REPO)

if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    subprocess.run(["git", "-C", REPO, "fetch", REPO_URL, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", REPO_URL, BRANCH], check=True)

os.chdir(REPO)
assert _has_script(), f"Manca {SCRIPT} — controlla repo/branch"
print("OK:", SCRIPT)
!git -C {REPO} remote -v | head -n 2
!git -C {REPO} rev-parse --short HEAD

In [ ]:
# 3) Dipendenze + build CUDA (T4 = sm_75)
import os
REPO = "/content/gnss_gpu"
os.chdir(REPO)

!pip install -q numpy matplotlib folium branca pyproj rasterio requests scipy
!apt-get -qq install -y cmake
!mkdir -p {REPO}/build
%cd {REPO}/build
!cmake .. -DCMAKE_CUDA_ARCHITECTURES=75
!make -j$(nproc)
%cd {REPO}

In [ ]:
# 4) Test import estensioni
import os, sys
REPO = "/content/gnss_gpu"
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
sys.path[:0] = [f"{REPO}/python", f"{REPO}/build"]
import gnss_gpu._bvh
print("_bvh OK")

In [ ]:
# 5) Setup pipeline (copia RINEX da experiments/data/TERF se presenti)
import os, subprocess
REPO = "/content/gnss_gpu"
SCRIPT = f"{REPO}/experiments/run_terf_permanent_station_colab.py"
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
subprocess.run(["mkdir", "-p", "/content/terf_work/data"], check=True)
subprocess.run(["python", SCRIPT, "--phase", "setup", "--work-dir", "/content/terf_work"], check=True)

In [ ]:
# 5b) Upload manuale TERF*.rnx (solo se non già nel repo)
from google.colab import files
from pathlib import Path

dest = Path("/content/terf_work/data")
dest.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    (dest / name).write_bytes(data)
    print("saved", dest / name)

In [ ]:
# 6) Labels geometriche (mesh + BRDC + CSV)
import os, subprocess
REPO = "/content/gnss_gpu"
SCRIPT = f"{REPO}/experiments/run_terf_permanent_station_colab.py"
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"

subprocess.run([
    "python", SCRIPT,
    "--work-dir", "/content/terf_work",
    "--phase", "all",
    "--systems", "G,C",
    "--epoch-step", "1",
], check=True)

In [ ]:
# 7) Token Cesium ion
import os
os.environ["CESIUM_ION_TOKEN"] = "PASTE_YOUR_CESIUM_ION_TOKEN_HERE"

In [ ]:
# 8) Viz 3D per-epoch (solo satelliti nel RINEX OBS)
import os, subprocess
REPO = "/content/gnss_gpu"
SCRIPT = f"{REPO}/experiments/run_terf_permanent_station_colab.py"
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"

subprocess.run([
    "python", SCRIPT,
    "--work-dir", "/content/terf_work",
    "--phase", "viz",
    "--viz-day", "2026192",
    "--n-epochs-viz", "24",
    "--epoch-min-interval-s", "600",
    "--obs-match-tol-s", "20",
], check=True)

In [ ]:
# 9) Apri viewer HTML
from pathlib import Path
from IPython.display import IFrame
import subprocess, time

html = Path("/content/terf_work/results/terf_los_viz_2026192.html")
if not html.exists():
    raise FileNotFoundError(html)

subprocess.Popen(
    "python -m http.server 8765 --directory /content/terf_work/results",
    shell=True,
)
time.sleep(1)
IFrame(src="http://localhost:8765/terf_los_viz_2026192.html", width="100%", height=700)

In [ ]:
# 10) Scarica zip
!zip -r /content/terf_results.zip /content/terf_work/results
from google.colab import files
files.download("/content/terf_results.zip")